# Serialization format to store and share simulations

HELIOS++ models, i.e., surveys, scenes, settings, etc., can be serialized to YAML files and to bundle directories. This is useful for archiving a simulation setup, sharing it with colleagues, or simply reproducing a result later. What makes HELIOS++ serialization special is that it does not just dump field values: it also records the *provenance* of an object, i.e., how it was create (e.g. `from_obj`) and which operations were applied afterwards (e.g. `translate`). Loading a YAML file therefore replays these steps rather than just restoring plain data.

## YAML

Serialization to YAML via `.to_yaml()` results in either
- a single file, in which all nested components are contained (default; `to_yaml(...)`, equivalent to `to_yaml(..., shallow=False)`), or
- a short root file plus separate YAML files for each nested component, linked together by references (`to_yaml(..., shallow=True)`).

We demonstrate this below by serializing a survey which refers to several other models such as the scene, platform, scanner, and various settings.

Before that, we import the required packages and create an exemplary helios survey.

In [1]:
import helios
import yaml
import pprint
import tempfile

In [25]:
# helper function to show YAML files
def show_yaml(yaml_path):
    with open(yaml_path, "r") as f:
        yaml_content = yaml.safe_load(f)
        pprint.pp(yaml_content)

In [3]:
helios.add_asset_directory("../data/sceneparts")
# create a survey for demonstration
platform = helios.platform_from_name("sr22")
scanner = helios.scanner_from_name("leica_als50")
sceneparts = helios.ScenePart.from_objs("toyblocks/*.obj")
additional_scene_part = helios.ScenePart.from_obj("toyblocks/cube.obj").translate([1, 1, 1])
scene = helios.StaticScene(sceneparts + [additional_scene_part])
survey = helios.Survey(scene=scene, scanner=scanner, platform=platform)
scan_settings = helios.ScannerSettings(
    pulse_frequency="100000 Hz",
    scan_angle="20 deg",
    scan_frequency="70 Hz",
    trajectory_time_interval="0.01 s",
)
waypoints = [[-50, -25, 250], [50, -25, 250]]
speed = 30  # m/s
for x, y, z in waypoints:
    survey.add_leg(x=x, y=y, z=z, speed_m_s=speed, scanner_settings=scan_settings)

The survey has now been created. We now serialize it to YAML files using both the inlined and the shallow version.

In [4]:
inlined_yaml = survey.to_yaml(tempfile.TemporaryFile().name)
shallow_yaml = survey.to_yaml(tempfile.TemporaryFile().name, shallow=True)

Let's first investigate the inlined YAML file. You can see that all information of the survey is contained in this file. Feel free to click on the link to the `serialization_schema.json`. This [JSON schema](https://json-schema.org/) documents the HELIOS++ model representation and is used to validate YAML files loaded via `from_yaml()`. Every file also records its `serialization_major_version` and `serialization_minor_version`, so HELIOS++ can tell whether it is compatible with a file written by a different HELIOS++ version (see the summary at the end of this notebook for details).

In [37]:
print(inlined_yaml.name)
show_yaml(inlined_yaml)

survey.yaml
{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.survey.Survey',
 'fields': {'scanner': {'serialization_major_version': 0,
                        'serialization_minor_version': 0,
                        'model_class': 'helios.scanner.Scanner',
                        'fields': {},
                        'provenance': {'constructor': {'method': 'from_xml',
                                                       'kwargs': {'scanner_file': 'scanners_als.xml',
                                                                  'scanner_id': 'leica_als50'}},
                                       'operations': []}},
            'platform': {'serialization_major_version': 0,
                         'serialization_minor_version': 0,
                         'model_class': 'helios.platforms.Platform',
         

Next, we compare the inlined YAML file with the shallow one. We see that the `survey.yaml` that we serialized is very short, but it contains multiple `'model_ref'` objects which refer to other YAML files. These always have consistent naming according to the model class that they describe and - if there are multiple instances for the class - an incremental ID.

In [38]:
print(shallow_yaml.name)
show_yaml(shallow_yaml)

survey.yaml
{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.survey.Survey',
 'fields': {'scanner': {'model_ref': 'scanner.yaml'},
            'platform': {'model_ref': 'platform.yaml'},
            'scene': {'model_ref': 'staticscene.yaml'},
            'legs': [{'model_ref': 'leg.yaml'}, {'model_ref': 'leg_2.yaml'}],
            'name': '',
            'gps_time': '2026-07-22T15:13:03.640982Z',
            'full_waveform_settings': {'model_ref': 'fullwaveformsettings.yaml'}}}


We can find all these files in the same directory. Actually, there are even more YAML files, since we have several nesting levels. For example, the survey YAMl references two YAML files for the legs which each link to additional YAML files for `dynamicplatformsettings` and `scannersettings`. 

In [ ]:
for fpath in shallow_yaml.parent.glob("*"):
    print(fpath.name)

dynamicplatformsettings.yaml
dynamicplatformsettings_2.yaml
fullwaveformsettings.yaml
leg.yaml
leg_2.yaml
platform.yaml
scanner.yaml
scannersettings.yaml
scannersettings_2.yaml
scenepart.yaml
scenepart_2.yaml
scenepart_3.yaml
scenepart_4.yaml
scenepart_5.yaml
scenepart_6.yaml
staticscene.yaml
survey.yaml
trajectorysettings.yaml
trajectorysettings_2.yaml


If we look into one of the scene part YAML files, we see that in the `provenance` section, it preserves
- the way the scene part was loaded (`'from_obj'`)
- the relative file path from which the scene part was loaded (`'obj_file': '../data/sceneparts/toyblocks/cube.obj'`)
- any operations performed (in this case, a `translate` operation)

However, note if we want a fully portable serialization that also contains scanner and platform specifications or scene part files, we need to use the `to_bundle()` methods as demonstrated in the next section.

In [28]:
show_yaml(shallow_yaml.parent / "scenepart_6.yaml")

{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.scene.ScenePart',
 'fields': {'id': None, 'force_on_ground': 0},
 'provenance': {'constructor': {'method': 'from_obj',
                                'kwargs': {'obj_file': '../data/sceneparts/toyblocks/cube.obj',
                                           'up_axis': 'z',
                                           'id': None}},
                'operations': [{'method': 'translate',
                                'kwargs': {'offset': [1.0, 1.0, 1.0]}}]}}


Loading works the other way around: `from_yaml()` reconstructs an object by replaying its recorded provenance, i.e., it calls the same constructor and the same sequence of operations that were originally used to build it. Any relative file paths it needs along the way (such as the `.obj` files of our scene parts) are resolved using the asset directories registered by default (current working directory, `helios` installation directory) or explicitly with `helios.add_asset_directory()`, so make sure these are set up before loading.

Since all information is stored, one call is enough to load the survey again from our generated YAML. Below, we compare the two waypoints to show that the reloaded survey matches the original:

In [29]:
survey_loaded = helios.Survey.from_yaml(inlined_yaml)

original_waypoints = [(leg.platform_settings.x, leg.platform_settings.y) for leg in survey.legs]
loaded_waypoints = [(leg.platform_settings.x, leg.platform_settings.y) for leg in survey_loaded.legs]
print(f"Original waypoints: {original_waypoints}")
print(f"Loaded waypoints:   {loaded_waypoints}")

Original waypoints: [(-50.0, -25.0), (50.0, -25.0)]
Loaded waypoints:   [(-50.0, -25.0), (50.0, -25.0)]


## Bundle

Serialization to bundle via `.to_bundle()` does not only create a shallow YAML file, but additionally writes provenance file references, i.e., it copies all used assets to the bundle. Let's demonstrate this for the same survey as before.

In [11]:
bundle_yaml = survey.to_bundle(path=tempfile.TemporaryDirectory().name)
bundle_dir = bundle_yaml.parent

Now if we look into this directory, we see not only YAML files, but also the `platforms.xml` and `scanners_als.xml` as well as all the `.obj` files in the scene and their linked `.mtl` files.
This bundle directory could now be zipped and uploaded to a data repository to openly share the fully portable, reproducible simulation data.

In [13]:
for fpath in bundle_dir.glob("*"):
    print(fpath.name)

cube.mtl
cube.obj
cube_rot.mtl
cube_rot.obj
cylinder.mtl
cylinder.obj
dynamicplatformsettings.yaml
dynamicplatformsettings_2.yaml
easter2021.mtl
easter2021.obj
fullwaveformsettings.yaml
leg.yaml
leg_2.yaml
platform.yaml
platforms.xml
scanner.yaml
scannersettings.yaml
scannersettings_2.yaml
scanners_als.xml
scenepart.yaml
scenepart_2.yaml
scenepart_3.yaml
scenepart_4.yaml
scenepart_5.yaml
scenepart_6.yaml
sphere.mtl
sphere.obj
staticscene.yaml
survey.yaml
trajectorysettings.yaml
trajectorysettings_2.yaml


If we again look at the scenepart YAML file, we now see that the `'obj_file'` field has only the file name rather than the previous relative path since the OBJ file is bundled directly inside the bundle directory.

In [30]:
show_yaml(bundle_dir / "scenepart_6.yaml")

{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.scene.ScenePart',
 'fields': {'id': None, 'force_on_ground': 0},
 'provenance': {'constructor': {'method': 'from_obj',
                                'kwargs': {'obj_file': 'cube.obj',
                                           'up_axis': 'z',
                                           'id': None}},
                'operations': [{'method': 'translate',
                                'kwargs': {'offset': [1.0, 1.0, 1.0]}}]}}


We can easily load the survey again from the bundle, without needing any asset directories to be set up since everything is contained in the bundle directory itself:

In [31]:
loaded_survey_from_bundle = helios.Survey.from_bundle(path=bundle_dir)

loaded_waypoints = [(leg.platform_settings.x, leg.platform_settings.y) for leg in loaded_survey_from_bundle.legs]
print(f"Loaded waypoints: {loaded_waypoints}")

Loaded waypoints: [(-50.0, -25.0), (50.0, -25.0)]


The bundle method has two additional boolean parameters, which are False by default.

- `binary=True` will use binary sidecar serialization where supported. Currently, this is the case for the scene. We demonstrate this below.
- `force=True` will clear the specified directory bundle if it is not empty. If `force=False` (default), then trying to write to an existing directory bundle will raise a `RuntimeError`.

In [36]:
bundle_yaml_binary = survey.to_bundle(path=tempfile.TemporaryDirectory().name,
                                      binary=True)
bundle_dir_binary = bundle_yaml_binary.parent
show_yaml(bundle_dir_binary / "staticscene.yaml")
for fpath in bundle_dir_binary.glob("*"):
    print(fpath.name)

{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.scene.StaticScene',
 'fields': {},
 'binary': {'method': 'from_binary', 'path': 'staticscene.bin'}}
dynamicplatformsettings.yaml
dynamicplatformsettings_2.yaml
fullwaveformsettings.yaml
leg.yaml
leg_2.yaml
platform.yaml
platforms.xml
scanner.yaml
scannersettings.yaml
scannersettings_2.yaml
scanners_als.xml
staticscene.bin
staticscene.yaml
survey.yaml
trajectorysettings.yaml
trajectorysettings_2.yaml


Now when we look at the `staticscene.yaml` and the bundle content, we can see that the scene parts are not written individually, but the entire scene is written to a binary `staticscene.bin`. When calling `from_bundle()`, this scene will be loaded via the `from_binary()` method.

As stated in the beginning, we can serialize any HELIOS++ model, not just a whole survey. For instance, we could also serialize just the scene:

In [32]:
scene_bundle_yaml = scene.to_bundle(path=tempfile.TemporaryDirectory().name)
scene_bundle_dir = scene_bundle_yaml.parent

In [33]:
for path in scene_bundle_dir.glob("*"):
    print(path.name)

cube.mtl
cube.obj
cube_rot.mtl
cube_rot.obj
cylinder.mtl
cylinder.obj
easter2021.mtl
easter2021.obj
scenepart.yaml
scenepart_2.yaml
scenepart_3.yaml
scenepart_4.yaml
scenepart_5.yaml
scenepart_6.yaml
sphere.mtl
sphere.obj
staticscene.yaml


In [34]:
show_yaml(scene_bundle_dir / "staticscene.yaml")
show_yaml(scene_bundle_dir / "scenepart.yaml")

{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.scene.StaticScene',
 'fields': {'scene_parts': [{'model_ref': 'scenepart.yaml'},
                            {'model_ref': 'scenepart_2.yaml'},
                            {'model_ref': 'scenepart_3.yaml'},
                            {'model_ref': 'scenepart_4.yaml'},
                            {'model_ref': 'scenepart_5.yaml'},
                            {'model_ref': 'scenepart_6.yaml'}]}}
{'$schema': 'https://raw.githubusercontent.com/3dgeo-heidelberg/helios/alpha-dev/python/helios/data/serialization_schema.json',
 'serialization_major_version': 0,
 'serialization_minor_version': 0,
 'model_class': 'helios.scene.ScenePart',
 'fields': {'id': None, 'force_on_ground': 0},
 'provenance': {'constructor': {'method': 'from_obj',
                                'kwargs

## Summary

To recap, here is a rule of thumb for choosing between the two serialization methods:

| Method | Result | Portable across machines? | Typical use case |
| --- | --- | --- | --- |
| `to_yaml(shallow=False)` (default) | one self-contained YAML file | only if all referenced files (`.obj`, `.xml`, ...) stay in place | quickly inspecting or diffing a single model |
| `to_yaml(shallow=True)` | one YAML file per model, linked by `model_ref` | only if all referenced files (`.obj`, `.xml`, ...) stay in place | keeping a readable, version-controllable set of configuration files |
| `to_bundle()` | a directory with YAML files plus copies of every referenced asset | yes, the directory is fully self-contained | archiving or sharing a simulation so others can reproduce it exactly |

A few more things worth knowing:
- Every serialized document carries a `serialization_major_version` and `serialization_minor_version`, and is validated against the [JSON schema](https://json-schema.org/) referenced in `$schema`. This means HELIOS++ can detect and, within the same major version, automatically migrate YAML files written by older versions of the format.
- `to_bundle()` also accepts `binary=True`, which stores supported models (such as scenes) as a compact binary sidecar file instead of re-exporting all source assets. This can be useful for large scenes where re-serializing to OBJ would be slow or bulky.
- Since serialization works on any `helios.Model` subclass, you are not limited to whole surveys: as shown above, individual components like a `StaticScene` can be serialized and shared on their own.